# 06 · Model Validation
Four tests: a pooled backtest of the forecast, an early-warning test of the score, a persona ground-truth check, and weight sensitivity.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
display(Markdown(open(ROOT / "reports/model_validation.md").read()))

# Model Validation

Generated by `src/validate_model.py`.

## 1. Cash-flow forecast backtest (720 customer-month forecasts, target months 2025-04..2025-09)

Each month's forecast uses only data up to the previous month (no look-ahead).

| Metric | Value |
|---|---:|
| Income forecast error (weighted MAPE) | 10.3% |
| Outflow forecast error (weighted MAPE) | 14.3% |
| Median abs. error of ending balance | ₹7,139 |
| ...as share of monthly income | 9.2% |
| Customer-months with an actual cash crunch* | 27 |
| Flagged High/Medium and had a crunch (TP) | 18 |
| Flagged but no crunch (FP) | 32 |
| Missed (FN) | 9 |
| Precision / Recall | 36% / 67% |
| AUC of deficit probability | 0.96 |
| Base rate of cash crunch | 3.8% |

Flagging = forecast tier High or Medium. The safety-cushion level (25% of expected outflows) was chosen on this same window, so these figures are slightly optimistic; a longer history would allow a clean out-of-time test.

*Cash crunch = the customer had to borrow during the month, or would have ended below zero without borrowing.

| Forecast tier | forecasts | had cash crunch | crunch rate |
|---|---|---|---|
| High | 27 | 12 | 44% |
| Medium | 23 | 6 | 26% |
| Low | 670 | 9 | 1% |

## 2. Early-warning test: stress band in month m vs borrowing in month m+1 (2025-03 to 2025-08)

| Stress band (month m) | customer-months | borrowed next month | borrow rate |
|---|---|---|---|
| Low | 461 | 0 | 0.0% |
| Moderate | 153 | 7 | 4.6% |
| High | 102 | 17 | 16.7% |
| Critical | 4 | 3 | 75.0% |

AUC of stress score for next-month borrowing: **0.93**

## 3. Persona check (ground truth only available because the data is synthetic)

AUC separating at-risk personas from healthy personas: **0.94**

| Hidden persona | Low | Moderate | High | Critical |
|---|---|---|---|---|
| comfortable_spender | 23 | 1 | 0 | 0 |
| emi_burdened | 0 | 12 | 5 | 1 |
| irregular_earner | 6 | 8 | 4 | 0 |
| lifestyle_creep | 5 | 7 | 0 | 0 |
| shock_event | 8 | 2 | 2 | 0 |
| stable_saver | 30 | 0 | 0 | 0 |
| student_early_career | 0 | 5 | 1 | 0 |

Mean score by persona: stable_saver 5.0, comfortable_spender 8.8, shock_event 27.7, lifestyle_creep 31.5, irregular_earner 41.6, emi_burdened 55.5, student_early_career 56.6

## 4. Weight sensitivity (benchmark weights vs equal weights)

- Customers keeping the same band: **92%**
- Rank correlation of scores: **0.98**

| Weighted band \ Equal-weight band | Low | Moderate | High | Critical |
|---|---|---|---|---|
| Low | 70 | 2 | 0 | 0 |
| Moderate | 4 | 31 | 0 | 0 |
| High | 0 | 1 | 9 | 2 |
| Critical | 0 | 0 | 1 | 0 |

Interpretation: rankings are stable to the weighting choice, so conclusions about *who* is most stressed do not depend on the exact weights; band counts shift mainly at boundaries.


In [3]:
ew = pd.read_csv(ROOT / "data/analytics/validation_early_warning.csv")
ew.groupby("stress_band")["borrowed_this_month"].agg(["count","mean"]).reindex(["Low","Moderate","High","Critical"]).rename(columns={"mean":"next-month borrow rate"})

,count,next-month borrow rate
stress_band,,
Low,461,0.00
Moderate,153,0.05
High,102,0.17
Critical,4,0.75
